# PD vs CTRL classifier from scRNA-seq (donor-level)

**Goal:** test whether a Parkinson's-disease signal is recoverable from PBMC transcriptomes, first at
whole-donor level and then per cell type, in a way that is *leakage-safe* and *not just detecting batch*.

## Design decisions (defend these in the methods chapter)

1. **Unit of analysis = the donor, not the cell.** The label (`diagnosis`) only varies at the donor
   level, so cells from one donor are pseudoreplicates. We aggregate cells into one **pseudobulk**
   profile per donor, so every row of the model is one independent-ish sample. n = 103 donors.

2. **Batch/dataset confounding is the main threat.** In pooled public cohorts `diagnosis` is often
   partly confounded with `dataset`. We (a) inspect the `diagnosis x dataset` cross-tab up front and
   (b) evaluate with **leave-one-dataset-out** CV: if the model still predicts an unseen *dataset*,
   the signal generalises across batches rather than being batch itself.

3. **Two feature sets, compared.** (a) Raw-count **pseudobulk** (log-CPM) built here from counts, and
   (b) the **scVI-corrected expression matrix** you derived (`integrated_adata_scvi_corrected_matrix.h5ad`),
   aggregated to donor level by mean. Both are gene-level and interpretable. The corrected matrix removes
   dataset effects, but because scVI conditions on `dataset` it can also remove disease signal that
   co-varies with dataset (over-correction). If the two feature sets disagree, that gap is a finding.

4. **Small n => simple, regularised models.** L1/L2 logistic regression with feature selection done
   *inside* the CV loop. No deep learning; it would overfit 103 samples instantly.

5. **Honesty checks:** report balanced accuracy / ROC-AUC / PR-AUC across folds (not raw accuracy),
   and a **label-shuffle null** to confirm performance collapses to chance when labels are permuted.



## 0. Imports & configuration


In [1]:
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_predict,
                                     LeaveOneGroupOut, permutation_test_score)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                             average_precision_score, roc_curve, confusion_matrix)

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 1
RNG = 0  # global random_state for reproducibility

In [4]:
# ---- CONFIG: adjust to match your integrated object -------------------------
ADATA_PATH   = "/rds/general/user/ztb25/home/PBMC_datasets/integrated_adata_scANVI.h5ad"

COL_SAMPLE    = "sample_combined"                     # donor-level ID
COL_DIAGNOSIS = "diagnosis"                           # "PD" / "CTRL"
COL_DATASET   = "dataset"                             # "ds1".."ds4"
COL_CELLTYPE  = "celltypist_cell_label_coarse"        # 8 coarse labels

POS_LABEL   = "PD"     # positive class
NEG_LABEL   = "CTRL"

# Where the RAW COUNTS live. Pseudobulk must be built from counts, NOT from
# log-normalised or scaled X. Set the one that is true for your object:
COUNTS_LAYER = "counts"   # e.g. adata.layers["counts"]; set to None to use adata.raw.X or adata.X

# scVI-corrected expression matrix (gene-level, derived by you) for Section 7. Set to None to skip.
CORRECTED_PATH   = "/data/processed/integrated_adata_scvi_corrected_matrix.h5ad"
CORRECTED_IS_LOG = False  # True if that matrix is already log-scaled; if False we log1p it

MIN_CELLS_PER_DONOR = 50   # drop donors too sparse to pseudobulk reliably
N_TOP_GENES         = 2000 # HVG cap fed into the model (feature selection refines inside CV)

## 1. Load & sanity-check the confounding structure

Before any modelling: is PD/CTRL actually separable *within* datasets, or is each dataset almost
all one class? If the latter, a cross-dataset classifier is really a dataset detector — note it here.


In [8]:
adata

import anndata as ad

In [9]:
# Bringing back diagnosis column, needed for composition analysis, that got dropped at concatenation.

# the per-dataset files that DO have diagnosis
files = {
    "1": "/rds/general/user/ztb25/home/PBMC_datasets/1/PBMC1_clustering.h5ad",
    "4": "/rds/general/user/ztb25/home/PBMC_datasets/4/PBMC4_clustering.h5ad",
    "5": "/rds/general/user/ztb25/home/PBMC_datasets/5/PBMC5_clustering.h5ad",
    "6": "/rds/general/user/ztb25/home/PBMC_datasets/6/PBMC6_clustering.h5ad",
}

# build a barcode -> diagnosis lookup, reconstructing the integrated barcode format
diag_map = {}
for ds, path in files.items():
    a = ad.read_h5ad(path, backed="r")
    integrated_barcodes = a.obs_names + f"-{ds}"
    if "diagnosis" in a.obs.columns:
        diags = a.obs["diagnosis"].astype(str).tolist()
    else:
        diags = ["CTRL"] * a.n_obs          # dataset 6 — no column, all controls
    diag_map.update(dict(zip(integrated_barcodes, diags)))
    a.file.close()

adata.obs["diagnosis"] = adata.obs_names.map(diag_map)

In [10]:
adata = sc.read_h5ad(ADATA_PATH)
print(adata)

# keep only the two classes of interest (drops AD / other if present)
adata = adata[adata.obs[COL_DIAGNOSIS].isin([POS_LABEL, NEG_LABEL])].copy()

# donor-level metadata table (one row per donor)
donor_meta = (adata.obs[[COL_SAMPLE, COL_DIAGNOSIS, COL_DATASET]]
              .drop_duplicates(COL_SAMPLE)
              .set_index(COL_SAMPLE))
print("\nDonors per class:")
print(donor_meta[COL_DIAGNOSIS].value_counts())

print("\ndiagnosis x dataset (donor counts) -- watch for confounding:")
ct = pd.crosstab(donor_meta[COL_DATASET], donor_meta[COL_DIAGNOSIS])
print(ct)

: 

: 

: 

In [1]:
# cells per donor -- flag donors below the pseudobulk threshold
cells_per_donor = adata.obs[COL_SAMPLE].value_counts()
print("cells/donor: min={}, median={}, max={}".format(
    cells_per_donor.min(), int(cells_per_donor.median()), cells_per_donor.max()))
low = cells_per_donor[cells_per_donor < MIN_CELLS_PER_DONOR]
print("donors below MIN_CELLS_PER_DONOR ({}): {}".format(MIN_CELLS_PER_DONOR, list(low.index)))

NameError: name 'adata' is not defined

## 2. Build the whole-donor pseudobulk matrix

Sum **raw counts** across all of a donor's cells, then log-CPM normalise. Summing (not averaging)
is the standard pseudobulk aggregation; CPM makes donors with different total cell counts comparable.


In [ ]:
from scipy.sparse import issparse, csr_matrix

def get_counts_matrix(ad):
    """Return a cells x genes count matrix from the configured source."""
    if COUNTS_LAYER is not None and COUNTS_LAYER in ad.layers:
        M = ad.layers[COUNTS_LAYER]
    elif ad.raw is not None:
        M = ad.raw.X
    else:
        M = ad.X
    return csr_matrix(M) if not issparse(M) else M.tocsr()

def build_pseudobulk(ad, genes=None):
    """donors x genes: summed counts -> log1p(CPM). Returns (DataFrame, gene_index)."""
    if genes is not None:
        ad = ad[:, genes]
    counts = get_counts_matrix(ad)
    donors = ad.obs[COL_SAMPLE].values
    uniq = pd.Index(pd.unique(donors))
    # sum counts per donor via an indicator matrix (donors x cells) @ (cells x genes)
    codes = uniq.get_indexer(donors)
    ind = csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))),
                     shape=(len(uniq), ad.n_obs))
    summed = np.asarray((ind @ counts).todense())
    lib = summed.sum(1, keepdims=True); lib[lib == 0] = 1
    logcpm = np.log1p(summed / lib * 1e6)
    return pd.DataFrame(logcpm, index=uniq, columns=ad.var_names), ad.var_names

# restrict to highly variable genes to keep the feature space sane
sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
                            layer=COUNTS_LAYER if (COUNTS_LAYER in adata.layers) else None)
hvg = adata.var_names[adata.var["highly_variable"]]
print("using", len(hvg), "HVGs")

X_df, _ = build_pseudobulk(adata, genes=hvg)
X_df = X_df.loc[cells_per_donor[cells_per_donor >= MIN_CELLS_PER_DONOR].index.intersection(X_df.index)]
meta = donor_meta.loc[X_df.index]
y = (meta[COL_DIAGNOSIS] == POS_LABEL).astype(int).values
groups_dataset = meta[COL_DATASET].values
print("pseudobulk matrix:", X_df.shape, "| PD =", int(y.sum()), "| CTRL =", int((1-y).sum()))

## 3. Whole-donor classifier

L2 logistic regression, `class_weight='balanced'` for any imbalance. **Feature selection is inside
the pipeline** so it is refit on each training fold only (selecting genes on all data first would leak).
We report two evaluations:
- **Stratified 5-fold** (each donor is independent, so plain StratifiedKFold on donors is valid here).
- **Leave-one-dataset-out** — the honest cross-batch test.


In [ ]:
def make_pipe(k="all", penalty="l2", C=1.0):
    return Pipeline([
        ("select", SelectKBest(f_classif, k=k)),
        ("scale",  StandardScaler()),
        ("clf",    LogisticRegression(penalty=penalty, C=C, class_weight="balanced",
                                      solver="liblinear", max_iter=2000, random_state=RNG)),
    ])

def evaluate(X, y, groups, cv, label):
    pipe = make_pipe(k=min(500, X.shape[1]))
    proba = cross_val_predict(pipe, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
    pred = (proba >= 0.5).astype(int)
    print("== {} ==".format(label))
    print("  balanced acc: {:.3f}".format(balanced_accuracy_score(y, pred)))
    print("  ROC-AUC:      {:.3f}".format(roc_auc_score(y, proba)))
    print("  PR-AUC:       {:.3f}".format(average_precision_score(y, proba)))
    print("  confusion [ [TN FP][FN TP] ]:\n", confusion_matrix(y, pred))
    return proba

X = X_df.values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
proba_skf = evaluate(X, y, None, skf, "Whole-donor | Stratified 5-fold")

logo = LeaveOneGroupOut()
print()
proba_logo = evaluate(X, y, groups_dataset, logo, "Whole-donor | Leave-one-dataset-out")

## 4. Label-shuffle null (leakage sanity check)

Permute labels *within the CV structure* and re-score. A real signal should sit well above the null;
if the true score is inside the null distribution, the model has nothing (or leakage inflated it).


In [ ]:
score, perm_scores, pval = permutation_test_score(
    make_pipe(k=min(500, X.shape[1])), X, y,
    scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=RNG),
    n_permutations=200, random_state=RNG, n_jobs=-1)
print("true ROC-AUC = {:.3f} | null mean = {:.3f} | p = {:.3f}".format(
    score, perm_scores.mean(), pval))

plt.figure(figsize=(5,3))
plt.hist(perm_scores, bins=25, alpha=.7, label="shuffled-label null")
plt.axvline(score, color="r", lw=2, label="true AUC")
plt.xlabel("ROC-AUC"); plt.ylabel("count"); plt.legend(); plt.title("Permutation null"); plt.show()

## 5. Which genes drive the call?

Refit on all donors (for interpretation only — performance numbers come from the CV above) and read
the largest-magnitude coefficients. Sign tells you up/down in PD.


In [ ]:
final = make_pipe(k=min(500, X.shape[1]))
final.fit(X, y)
sel_mask = final.named_steps["select"].get_support()
sel_genes = X_df.columns[sel_mask]
coefs = final.named_steps["clf"].coef_.ravel()
coef_tbl = (pd.Series(coefs, index=sel_genes)
            .sort_values(key=np.abs, ascending=False).head(30))
print("Top genes (positive = up in PD):")
print(coef_tbl)

## 6. Per-cell-type classifiers

Rebuild pseudobulk *within each coarse cell type* and classify separately. This localises the signal:
e.g. is PD separability driven by monocytes vs T cells? Cell types with too few cells in some donors
will be unstable — the per-type donor coverage is printed so you can discount thin ones.


In [ ]:
results = []
for ctype in sorted(adata.obs[COL_CELLTYPE].unique()):
    sub = adata[adata.obs[COL_CELLTYPE] == ctype]
    # donors that have enough cells of THIS type
    cpd = sub.obs[COL_SAMPLE].value_counts()
    keep_donors = cpd[cpd >= MIN_CELLS_PER_DONOR].index
    if len(keep_donors) < 20:
        print("skip {:20s} (only {} donors with >= {} cells)".format(
            ctype, len(keep_donors), MIN_CELLS_PER_DONOR)); continue
    sub = sub[sub.obs[COL_SAMPLE].isin(keep_donors)]
    Xc_df, _ = build_pseudobulk(sub, genes=hvg)
    mc = donor_meta.loc[Xc_df.index]
    yc = (mc[COL_DIAGNOSIS] == POS_LABEL).astype(int).values
    if yc.sum() < 5 or (1-yc).sum() < 5:
        print("skip {:20s} (class too small)".format(ctype)); continue
    pipe = make_pipe(k=min(500, Xc_df.shape[1]))
    proba = cross_val_predict(pipe, Xc_df.values, yc,
                              cv=StratifiedKFold(5, shuffle=True, random_state=RNG),
                              method="predict_proba")[:, 1]
    auc = roc_auc_score(yc, proba); bacc = balanced_accuracy_score(yc, (proba>=.5).astype(int))
    results.append({"cell_type": ctype, "n_donors": len(yc),
                    "PD": int(yc.sum()), "ROC_AUC": auc, "bal_acc": bacc})
    print("{:20s} n={:3d}  AUC={:.3f}  balAcc={:.3f}".format(ctype, len(yc), auc, bacc))

res_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
res_df

In [ ]:
if len(res_df):
    plt.figure(figsize=(6, 0.4*len(res_df)+1))
    plt.barh(res_df["cell_type"], res_df["ROC_AUC"])
    plt.axvline(0.5, color="k", ls="--", lw=1)
    plt.xlabel("cross-validated ROC-AUC"); plt.title("PD vs CTRL separability per cell type")
    plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 7. Parallel model: scVI-corrected expression matrix

Your derived corrected matrix is gene-level, so we can run the *same* interpretable classifier on it
and compare to the raw-count pseudobulk (Section 3). Because the corrected values are denoised model
expression (not counts), we aggregate to donor level by **mean per gene** rather than summing + CPM.

How to read it against Section 3:
- **Corrected >> raw** on 5-fold but both drop under LODO -> raw model was partly learning batch; correction helped.
- **Raw >> corrected** -> scVI likely over-corrected and removed real PD signal confounded with dataset.
- **Both hold up under leave-one-dataset-out** -> strongest result; the signal is genuinely cross-cohort.


In [ ]:
def donor_mean_expr(ad, genes):
    """donors x genes: MEAN of (already-corrected) expression per donor."""
    ad = ad[:, [g for g in genes if g in ad.var_names]]
    M = ad.X
    M = M.toarray() if issparse(M) else np.asarray(M)
    if not CORRECTED_IS_LOG:
        M = np.log1p(M)
    df = pd.DataFrame(M, index=ad.obs_names, columns=ad.var_names)
    df[COL_SAMPLE] = ad.obs[COL_SAMPLE].values
    return df.groupby(COL_SAMPLE).mean()

if CORRECTED_PATH is not None:
    adata_corr = sc.read_h5ad(CORRECTED_PATH)
    adata_corr = adata_corr[adata_corr.obs[COL_DIAGNOSIS].isin([POS_LABEL, NEG_LABEL])].copy()
    Xc = donor_mean_expr(adata_corr, hvg)
    Xc = Xc.loc[Xc.index.intersection(X_df.index)]     # same donors as the raw model
    yc = (donor_meta.loc[Xc.index, COL_DIAGNOSIS] == POS_LABEL).astype(int).values
    gc = donor_meta.loc[Xc.index, COL_DATASET].values
    print("corrected matrix:", Xc.shape, "| genes overlap =", Xc.shape[1])

    p_skf  = cross_val_predict(make_pipe(k=min(500, Xc.shape[1])), Xc.values, yc,
                               cv=StratifiedKFold(5, shuffle=True, random_state=RNG),
                               method="predict_proba")[:,1]
    p_logo = cross_val_predict(make_pipe(k=min(500, Xc.shape[1])), Xc.values, yc,
                               cv=LeaveOneGroupOut(), groups=gc,
                               method="predict_proba")[:,1]
    print("\ncorrected expr | 5-fold  AUC = {:.3f}".format(roc_auc_score(yc, p_skf)))
    print("corrected expr | LODO    AUC = {:.3f}".format(roc_auc_score(yc, p_logo)))
    print("\n(compare with raw-count pseudobulk in Section 3)")
else:
    print("CORRECTED_PATH not set - skipping.")

## 8. Reading the results (for the write-up)

- **Trust leave-one-dataset-out over the 5-fold number.** If LODO AUC ~ 0.5 but 5-fold is high, the
  model mostly learned batch — say so explicitly rather than reporting the optimistic figure.
- **The label-shuffle null** is your guard against silent leakage; cite the p-value.
- **Per-cell-type panel** localises the biology — lead the discussion with the top cell types and
  cross-reference their top genes against your differential-expression results.
- **Caveats to state:** n=103 donors is small for cross-cohort generalisation (wide CIs); results are
  hypothesis-generating, not a clinical predictor; pseudobulk discards within-donor heterogeneity;
  gene- vs scANVI-model disagreement reflects the batch/disease-signal trade-off.
